# Повторение: анализ данных, основы ML

**Время:** около одна пара. **Максимум:** 30 баллов.

В каждом задании:
- код — в ячейке `# Ваш код`;
- текстовый ответ — в ячейке `# Ваш ответ` (строка или словарь);
- если есть пункт **Вывод:** — 1–2 предложения.

Можно добавлять ячейки. 

**Формулировки заданий не удаляйте!** 

Данные создаются в Setup: таблица `df` про школьников (сколько готовились, с какого устройства заходили) и факт сдачи зачёта `exam_pass`.


In [ ]:
# Setup
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split

sns.set_theme(style="whitegrid")
rng = np.random.default_rng(42)

n = 400
klass = rng.choice([9, 10, 11], size=n, p=[0.25, 0.45, 0.30])
hours = np.clip(rng.normal(4.5 + (klass - 9) * 0.8, 2.0, n), 0.2, 16).round(1)
n_tasks = rng.integers(2, 18, n)
device = rng.choice(
    ["ПК", "телефон", "планшет", "ПК ", "пк"],
    size=n,
    p=[0.42, 0.35, 0.10, 0.08, 0.05],
)
late_prev = rng.choice([0, 1], size=n, p=[0.72, 0.28])
comment_len = rng.integers(0, 400, n)
logit = -1.2 + 0.18 * hours + 0.06 * n_tasks - 0.9 * late_prev + 0.35 * (klass == 11)
exam_pass = (rng.random(n) < 1 / (1 + np.exp(-logit))).astype(int)
exam_score = np.clip(45 + 35 * exam_pass + rng.normal(0, 8, n), 0, 100).round(0)
teacher_note = np.where(exam_pass == 1, "зачёт", "незачёт")

df = pd.DataFrame({
    "id": np.arange(n),
    "klass": klass,
    "hours": hours,
    "n_tasks": n_tasks,
    "device": device,
    "late_prev": late_prev,
    "comment_len": comment_len,
    "exam_score": exam_score,
    "teacher_note": teacher_note,
    "exam_pass": exam_pass,
})
df.loc[rng.choice(n, 28, replace=False), "hours"] = np.nan

print("Готово. Строк:", len(df), "столбцов:", df.shape[1])
df.head()


Готово. Строк: 400 столбцов: 10


,id,klass,hours,n_tasks,device,late_prev,comment_len,exam_score,teacher_note,exam_pass
0,0,11,7.0,3,телефон,0,14,89.0,зачёт,1
1,1,10,6.3,16,планшет,0,304,77.0,зачёт,1
2,2,11,6.7,8,ПК,1,384,38.0,незачёт,0
3,3,10,2.5,10,телефон,1,189,51.0,незачёт,0
4,4,9,0.2,4,ПК,0,216,36.0,незачёт,0


### Столбцы `df`

| столбец | смысл |
|---|---|
| `id` | номер ученика |
| `klass` | класс: 9 / 10 / 11 |
| `hours` | часы подготовки (есть пропуски) |
| `n_tasks` | сколько задач сдал в четверти |
| `device` | устройство входа (есть «грязные» написания) |
| `late_prev` | 1, если раньше сдавал работы поздно |
| `comment_len` | длина комментария к домашнему заданию |
| `exam_score` | баллы за **уже прошедший** зачёт |
| `teacher_note` | пометка учителя **после** зачёта |
| `exam_pass` | **цель:** сдал зачёт (1) или нет (0) |

Представьте: вы строите модель **до** зачёта, чтобы понять, кому нужна помощь.


---
## Практика


### Задача 1

Посмотрите на данные: `shape`, типы, `head()`, число пропусков по столбцам (`isna().sum()`).

В **Выводе:** какие проблемы в таблице видны сразу (пропуски, типы, грязные категории — что заметили).


In [ ]:
# Ваш код


**Вывод:** *ваш текст здесь*


### Задача 2

По столбцу `device` посчитайте `value_counts()` (и доли через `normalize=True`).

Приведите написания к одному виду (пробелы, регистр) и снова напечатайте частоты. Исходный `df` можно изменить или сделать столбец `device_clean`.

В **Выводе:** сколько уникальных устройств было «на глаз» до очистки и сколько реальных категорий осталось.


In [ ]:
# Ваш код


**Вывод:** *ваш текст здесь*


### Задача 3

Постройте **два** графика (используйте `subplots`):
1. гистограмма `hours` (пропуски можно не заполнять — hist их игнорирует);
2. столбцы числа учеников по `exam_pass` (`countplot` или `value_counts().plot(kind="bar")`).

В **Выводе:** есть ли дисбаланс классов и что видно по часам подготовки.


In [ ]:
# Ваш код


**Вывод:** *ваш текст здесь*


### Задача 4

Цель модели — `exam_pass` **до** зачёта.

В ячейке ответа перечислите столбцы, которые **нельзя** брать в признаки (утечка или идентификатор без смысла). Кратко почему.

Затем в ячейке кода соберите `df_model`: удалите эти столбцы (и исходный грязный `device`, если уже есть `device_clean`). Напечатайте список оставшихся столбцов.

Подсказка: всё, что появляется **только после** зачёта, для прогноза «успеет ли сдать» — утечка.


In [ ]:
# Ваш ответ
"""перечислите столбцы и почему"""


'перечислите столбцы и почему'

In [ ]:
# Ваш код


### Задача 5

Разбейте `df_model` на train/test: `test_size=0.25`, `random_state=42`, **стратификация** по `exam_pass`.

Напечатайте число строк в train и test и доли класса 1 в обеих выборках.

Пропуски в `hours` на этом шаге можно заполнить медианой **train** и той же медианой — test (без подглядывания в test).

В **Выводе:** зачем стратификация и почему медиану считают только на train.


In [ ]:
# Ваш код


**Вывод:** *ваш текст здесь*


### Задача 6

Постройте **константный baseline** без обучения «умной» модели: предскажите для всех объектов test **самый частый класс train**.

Напечатайте accuracy этого правила на test. Сравните с долей класса 1 на test.

В **Выводе:** почему такую константу всегда считают до построения модели машинного обучения.


In [ ]:
# Ваш код


**Вывод:** *ваш текст здесь*


### Задача 7

Модель в чужом ноутбуке выдала такую матрицу ошибок на test (строки — истина, столбцы — прогноз):

```text
              pred 0    pred 1
true 0          32        12
true 1           8        48
```

В ячейке ответа напишите:
- сколько всего объектов;
- accuracy;
- для класса 1: recall (полнота) и precision (точность) — можно примерно, с формулой;
- кого модель чаще обижает: ложно «сдал» или ложно «не сдал».

Для вычисления указанных величин напишите код в Python.


In [ ]:
# Ваш ответ
"""accuracy, recall, precision, комментарий"""


'accuracy, recall, precision, комментарий'

### Задача 8

Ученик описывает пайплайн:

> «Я сначала нормирую все признаки **по всей выборке**, потом делаю `train_test_split` и учу модель. На test accuracy почти как на train».

Выберите **одну** главную ошибку и запишите букву в `answer_8`.

- **A.** Нужно было взять learning rate побольше.
- **B.** Статистики и словарь посчитаны с участием test — утечка, метрика завышена.
- **C.** `train_test_split` всегда портит accuracy, его нельзя использовать.
- **D.** Нормализация запрещена: признаки должны остаться в сырых единицах.

Ниже одним-двумя предложениями поясните выбор (как исправить).


In [ ]:
# Ваш ответ
answer_8 = ""
comment_8 = """пояснение"""


### Задача 9

Сопоставьте **вопрос к данным** и тип графика. Заполните словарь значениями из списка: `hist`, `bar`, `scatter` (каждое слово по одному разу).

```python
answer_9 = {
    "как распределены часы подготовки": "",
    "сколько учеников в 9, 10 и 11 классе": "",
    "есть ли связь часов подготовки и балла за зачёт": "",
}
```


In [ ]:
# Ваш ответ
answer_9 = {
    "как распределены часы подготовки": "",
    "сколько учеников в 9, 10 и 11 классе": "",
    "есть ли связь часов подготовки и балла за зачёт": "",
}


### Задача 10

Школьный сюжет.

На защите проекта студент показывает: «обучающая выборка, accuracy 0.99». Тест не показывал. Датасет маленький, признаки включают и ответ в зашифрованном виде / имя файла картинки содержит класс / в тексте отзыва есть слово «зачёт». Что не так с этим студенческим проектом?

Напишите 4–6 предложений: (1) что такое переобучение простыми словами; (2) почему одной метрики на train мало; (3) какой минимальный честный ритуал требуется при выполнении проекта по машинному обучению.


In [ ]:
# Ваш ответ
"""ваш текст"""


'ваш текст'

### Задача 11
Постройте одну линейную и одну нелинейную модель для предсказания сдаст студент зачет или нет. Оцените качество предсказаний с помощью accuracy и F1-score. Оцените качество построенных моделей по сравнению с baseline и между собой графически. Сформулируйте вывод по полученным результатам.

In [1]:
# Ваш код

**Вывод:** ваш текст